# Tutorial: Estimating Social Contact Matrices with SocialMix

In [ ]:
import numpy as np
import pandas as pd

from cntmosaic.utils import AgeBins
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import (
    ParticipantGenerator,
    MatrixGenerator,
    ContactGenerator,
    Stratification,
    PopulationConstructor,
)
from cntmosaic.dataloader import ContactData, ParticipantData, PopulationData
from cntmosaic.vis import plot_mosaic, plot_mosaic_pixilated

from cntmosaic.models import SocialMix
from cntmosaic.analysis import ModelSummariserSocialMix, ModelEvaluatorSocialMix

from cntmosaic.utils import pixilate

import altair as alt

## 1. Introduction
The `SocialMix` class in `cntmosaic` is a Python implementation of the `socialmixr` algorithm for estimating social contact matrices from survey data. This tutorial will guide you through the process of using `SocialMix` to analyze contact survey data.

### Relationship to socialmixr
The original `socialmixr` is an R package widely used for estimating social contact matrices from survey data. The `SocialMix` class in `cntmosaic` replicates the core functionalities of `socialmixr`, allowing users to perform similar analyses within the Python ecosystem. This includes the ability to estimate contact intensities, apply reciprocity adjustments, compute contact rates, and bootstrap procedures for quantifying uncertainty. `SocialMix` also implements an adaptive age group merging strategy that automatically combines age groups with no participants with adjacent groups to ensure that the algorithm functions correctly even when certain age groups lack participant data.

### Mathematical Foundation
The core estimation follows these steps:

1. **Aggregate contacts by age group pairs**:
   $$
   y_{c,d} = \sum_{i = 1}^N 1_c(c_i) y_{i,d}
   $$
   where $1_c(c_i)$ is an indicator function that equals 1 if contact $i$ belongs to age group $c$, and $y_{i,d}$ is the number of contacts reported by participant $i$ with age group $d$.

2. **Compute contact intensity**:
   $$
   \hat{m}_{c,d} = \frac{y_{c,d}}{N_c}
   $$
   where $N_c$ is the number of participants in age group $c$.

3. **Apply reciprocity adjustment**:
   $$
   \hat{m}_{c,d}^{\dagger} = \frac{1}{2P_c} \left( \hat{m}_{c,d} P_c + \hat{m}_{d,c} P_d \right)
   $$
   ensuring that $P_c m_{c,d} = P_d m_{d,c}$ (total contact from $c$ to $d$ equals contacts from $d$ to $c$).

4. **Compute contact rates**:
   $$
   \hat{\gamma}_{c,d} = \frac{\hat{m}_{c,d}}{P_d}
   $$
   where $P_d$ is the population size of age group $d$.

## 2. Synthetic Data Generation
We'll use `cntmosaic`'s data generation tools to create realistic synthetic contact data. For details, refer to the "Tutorial: Generating Synthetic Contact Data".

We first load the contact pattern templates and population data:

In [ ]:
templates = load_template_patterns(country="United_States")
df_age_dist = load_age_distribution(country="United_States")

age_dist = df_age_dist.P.values

We define a subgroup with 1000 participants with the `Subgroup` class:

In [ ]:
strat = Stratification(
    name="general",
    n_strata=1,
    labels=["All"],
    ref_age_dist=df_age_dist.P.values,
    seed=42,
)

# Build population constructor
popcon = PopulationConstructor(strat)
df_pop = popcon.df_P

# Generate contact matrices
matrix_gen = MatrixGenerator(templates)
cnt_matrix = matrix_gen.generate_single(strat, seed=3)

# Generate participants and contacts
part_gen = ParticipantGenerator(popcon, n_part=500)
df_part = part_gen.generate(seed=2)

cnt_gen = ContactGenerator(df_part, cint_matrices=cnt_matrix, model="poisson")
df_cnt = cnt_gen.generate(seed=2)

We randomly simulate a contact intensity matrix using the `MatrixGenerator` class.

Visualize the generated contact matrix with `plot_mosaic` function:

In [ ]:
plot_mosaic(
    cnt_matrix["All->All"], title="True Contact intensity matrix", zlabel="Intensity"
)

The `ParticipantGenerator` class is then used to simulate participant data based on the defined subgroup.

## 3. Estimating Contact Matrices with SocialMix

### 3.1 Data Preparation

Besides the participant, contact, and population age distribution dataframes, the `SocialMix` class requires information about the age bins used for grouping the participants and contacts. We give these information in the form of an `AgeBins` object. Here, we define 5-year age bins from 0 to 80 years. The participant dataframe must also contain a column named `age_part` (or `age_grp_part` if age groups are pre-defined).

In [ ]:
age_bins = AgeBins(0, 80, 5)
part_date = ParticipantData(df_part, id_col="id", age_col="age")
cnt_data = ContactData(df_cnt, id_col="id", age_col="age_cnt")
pop_data = PopulationData(df_pop, age_col="age", size_col="P")

sm = SocialMix(
    part_date,
    cnt_data,
    age_bins,
    pop_data,
    apply_reciprocity=True,
    adaptive_merge=False,
    validate_for_bootstrap=False,
)

### 3.2 Basic Matrix Estimation

In [ ]:
cint_est = sm.cint()

plot_mosaic_pixilated(
    cint_est["All->All"], age_bins, title="Estimated Contact Matrix", zlabel="Intensity"
)

#### Mathematical Details: Contact Intensity Estimation

The contact intensity matrix $M$ is computed as:
$$
M = N^{-1} Y
$$
where

- $Y \in \mathbb{R}^{B \times B}$ is the contact count matrix, with $Y_{c,d}$ representing the total number of contacts reported by participants in age group $c$ with contacts in age group $d$.
- $N \in \mathbb{R}^{B \times B}$ is a diagonal matrix where $N_{c,c}$ is the number of participants in age group $c$.
- $m_{c,d}$ represent the average number of contacts from age group $c$ to age group $d$.

### 3.3 Contact Rate Matrix

In [ ]:
# Compute contact rate matrix
rate_est = sm.rate()

plot_mosaic_pixilated(
    rate_est["All->All"], age_bins, title="Estimated Contact Rate Matrix", zlabel="Rate"
)

#### Mathematical Details: Contact Rates

The contact rate normalizes by target population size:
$$
\Gamma = M P^{-1}
$$
where $P = \text{diag}(P_1, \ldots, P_B)$ contains population sizes.

In [ ]:
boot = sm.run_inference_bootstrap(n_boot=3000, random_state=42)

### 3.6 Summarizing Bootstrap Results
Bootstrap results can be summarized using the `ModelSummariserSocialMix` class. This class provides methods to extract key statistics from the bootstrap samples, such as mean estimates, confidence intervals, and other relevant metrics.

In [ ]:
summariser = ModelSummariserSocialMix(sm)
summary = summariser.summarise_cint(alpha=0.05)

In [ ]:
cint_est = sm.cint()

plot_mosaic_pixilated(
    cint_est["All->All"], age_bins, title="Estimated Contact Matrix", zlabel="Intensity"
)

In [ ]:
plot_mosaic_pixilated(
    pixilate(cnt_matrix["All->All"], age_bins, df_pop["P"].values),
    age_bins,
    title="Estimated Contact Matrix",
    zlabel="Intensity",
)

In [ ]:
plt_lower = plot_mosaic_pixilated(
    summary[0], age_bins, title="Lower", zlabel="Intensity"
)
plt_median = plot_mosaic_pixilated(
    summary[1], age_bins, title="Median", zlabel="Intensity"
)
plt_upper = plot_mosaic_pixilated(
    summary[2], age_bins, title="Upper", zlabel="Intensity"
)

alt.hconcat(plt_lower, plt_median, plt_upper)

### 3.7 Evaluating Model Accuracy
The accuracy of the estimated contact matrices can be evaluated by comparing them to the true contact matrices used in the synthetic data generation. The `ModelEvaluatorSocialMix` class can be used to compute a wide range of accuracy metrics.

In [ ]:
# Initialize evaluator
evaluator = ModelEvaluatorSocialMix(summariser, cint_matrix_true=cnt_matrix)

In [ ]:
# Default metrics
evaluator.evaluate()

In [ ]:
# Create an unstratified population
strat = Stratification(
    name="sex",
    n_strata=2,  # Single stratum = unstratified
    ref_age_dist=age_dist,
    labels=["M", "F"],
    seed=42,
)

# Build population constructor
popcon = PopulationConstructor(strat)

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
# For unstratified, use generate_single() with mean_intensity parameter
contact_matrix = matrix_gen.generate_partial(
    popcon=popcon,
    mean_intensity=15.0,  # Average 15 contacts per person
    seed=42,  # For reproducibility
)

print("Number of matrices:", len(contact_matrix.keys()))
print("Keys:", contact_matrix.keys())
print("Matrix shape:", contact_matrix["M->All"].shape)
print("Average contacts per person:", contact_matrix["M->All"].sum(axis=0).mean())

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(popcon, n_part=1500)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part, cint_matrices=contact_matrix, model="poisson"
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

In [ ]:
# Create PopulationData instance for model summarization
df_pop = popcon.df_P

part_data = ParticipantData(df_part, id_col="id", age_col="age", strat_var_cols="sex")
cnt_data = ContactData(df_cnt, id_col="id", age_col="age_cnt")
pop_data = PopulationData(df_pop, age_col="age", size_col="P", strat_var_cols="sex")
age_bins = AgeBins(0, 80, 5)

In [ ]:
sm = SocialMix(
    part_data,
    cnt_data,
    age_bins,
    pop_data,
    apply_reciprocity=False,
    adaptive_merge=False,
    validate_for_bootstrap=False,
)

In [ ]:
boot = sm.run_inference_bootstrap(n_boot=3000, random_state=42)

In [ ]:
# Initialize summariser with PopulationData
summariser = ModelSummariserSocialMix(sm)

In [ ]:
# Test summarisation with stratified model
# For K=2 (sex stratification), output is Dict[str, NDArray]
# where keys are stratum labels and values are (3, A, A) arrays
summary_partial = summariser.summarise_cint(alpha=0.05)

print(f"Summary type: {type(summary_partial)}")
print(f"Keys: {list(summary_partial.keys())}")
print(f"Shape for 'M': {summary_partial['M->All'].shape}")

In [ ]:
chart_m_all = plot_mosaic_pixilated(
    summary_partial["M->All"][1],
    age_bins,
    title="Inferred Contact Intensity Matrix for Men (Median)",
    zlabel="Intensity",
    ylabel=None,
)

chart_f_all = plot_mosaic_pixilated(
    summary_partial["F->All"][1],
    age_bins,
    title="Inferred Contact Intensity Matrix for Women (Median)",
    zlabel="Intensity",
    ylabel=None,
)

alt.hconcat(chart_m_all, chart_f_all)

In [ ]:
evaluator = ModelEvaluatorSocialMix(summariser, contact_matrix)

In [ ]:
evaluator.evaluate()

In [ ]:
# Initialize the matrix generator
matrix_gen = MatrixGenerator(templates)

# Generate a contact intensity matrix
contact_matrix = matrix_gen.generate_full(
    popcon=popcon,
    mean_intensity=10.0,  # Average 10 contacts per person
    seed=42,  # For reproducibility
)

In [ ]:
# Initialize the participant generator
part_gen = ParticipantGenerator(popcon, n_part=500)

# Generate participants
df_part = part_gen.generate(seed=42)

print(df_part.head())

In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part, cint_matrices=contact_matrix, model="poisson"
)

df_cnt = cnt_gen.generate(seed=42)

print(f"Generated {len(df_cnt)} contacts")
print(df_cnt.head())

In [ ]:
# Create PopulationData instance for model summarization
df_pop = popcon.df_P

part_data = ParticipantData(df_part, id_col="id", age_col="age", strat_var_cols="sex")
# For cnt_data, use None to auto-detect since df_cnt already has sex_cnt column
cnt_data = ContactData(df_cnt, id_col="id", age_col="age_cnt", strat_var_cols="sex_cnt")
pop_data = PopulationData(df_pop, age_col="age", size_col="P", strat_var_cols="sex")
age_bins = AgeBins(0, 80, 5)

In [ ]:
sm = SocialMix(
    part_data,
    cnt_data,
    age_bins,
    pop_data,
    apply_reciprocity=True,
    adaptive_merge=True,
    validate_for_bootstrap=True,
)

In [ ]:
boot = sm.run_inference_bootstrap(n_boot=3000, random_state=42)

In [ ]:
# Initialize summariser with PopulationData
summariser = ModelSummariserSocialMix(sm)

In [ ]:
evaluator = ModelEvaluatorSocialMix(summariser, contact_matrix)

In [ ]:
evaluator.evaluate()